In [6]:
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ============================================================
# PATHS
# ============================================================
BASE_DIR = r"C:\Users\Alsha\Desktop\Last EXP 15-6-2026"

SUMMARY_CSV = os.path.join(BASE_DIR, "summary_both_frameworks.csv")
FL_ROUNDS_CSV = os.path.join(BASE_DIR, "per_round_MNIST_FL_100nodes_100rounds.csv")
DFL_ROUNDS_CSV = os.path.join(BASE_DIR, "per_round_MNIST_DFL_100nodes_100rounds.csv")

# If your files have different names, uncomment and edit these:
# SUMMARY_CSV = os.path.join(BASE_DIR, "summary_both_frameworks(1).csv")
# FL_ROUNDS_CSV = os.path.join(BASE_DIR, "per_round_MNIST_FL_100nodes_100rounds(3).csv")
# DFL_ROUNDS_CSV = os.path.join(BASE_DIR, "per_round_MNIST_DFL_100nodes_100rounds(3).csv")

OUT_DIR = os.path.join(BASE_DIR, "MNIST_Final_Figures_Journal_Style")
os.makedirs(OUT_DIR, exist_ok=True)

# Use your already-generated data-distribution figure.
# Put this PNG inside BASE_DIR, or edit the path below.
DATA_DIST_IMAGE = os.path.join(BASE_DIR, "MNIST_data_distribution_FL_DFL_100nodes.png")

# Optional fallback if image is not found.
DATA_DIST_CSV = os.path.join(
    BASE_DIR,
    "FINAL_MNIST_3MAY_DFL_100NODES_100ROUNDS",
    "figures",
    "MNIST_data_distribution_FL_DFL_100nodes.csv"
)

# ============================================================
# STYLE
# ============================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "-",
})

FL_COLOR = "#1f77b4"
DFL_COLOR = "#ff7f0e"
FL_MARKER = "o"
DFL_MARKER = "s"
FL_STYLE = "-"
DFL_STYLE = "--"

EPS_ORDER = ["NoDP", "ε=0.5", "ε=1", "ε=2", "ε=3", "ε=4"]
EPS_NUMERIC = [0.5, 1, 2, 3, 4]

# Same style for ALL figures: title first, legend below title, both outside the axes.
TITLE_Y = 1.16
LEGEND_Y = 1.12


# ============================================================
# HELPERS
# ============================================================
def save_fig(fig, filename):
    png = os.path.join(OUT_DIR, filename + ".png")
    pdf = os.path.join(OUT_DIR, filename + ".pdf")
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", png)
    print("Saved:", pdf)


def normalize_epsilon_label(dp, eps):
    if str(dp).lower() == "false" or pd.isna(eps) or str(eps).upper() == "NAN":
        return "NoDP"
    try:
        e = float(eps)
        if e.is_integer():
            return f"ε={int(e)}"
        return f"ε={e:g}"
    except Exception:
        return f"ε={eps}"


def label_from_row(row):
    return normalize_epsilon_label(row.get("DP", None), row.get("Epsilon", np.nan))


def to_numeric_or_nan(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str) and x.strip().lower() in ["nan", "none", "never", ""]:
        return np.nan
    try:
        return float(x)
    except Exception:
        return np.nan


def prepare_summary(summary):
    summary = summary.copy()
    summary["Setting Label"] = summary.apply(label_from_row, axis=1)

    text_cols = ["Framework", "Dataset", "Final Model Path", "Gradient Attack CSV", "Setting Label"]
    for col in summary.columns:
        if col not in text_cols:
            summary[col] = summary[col].apply(to_numeric_or_nan)

    auc_cols = ["Confidence Attack AUC", "Loss Attack AUC", "Entropy Attack AUC"]
    acc_cols = ["Confidence Attack Accuracy", "Loss Attack Accuracy", "Entropy Attack Accuracy"]

    if all(c in summary.columns for c in auc_cols):
        summary["Strongest MIA AUC"] = summary[auc_cols].max(axis=1)

    if all(c in summary.columns for c in acc_cols):
        summary["Strongest MIA Accuracy"] = summary[acc_cols].max(axis=1)

    return summary


def get_row(summary, framework, label):
    tmp = summary[(summary["Framework"] == framework) & (summary["Setting Label"] == label)]
    if len(tmp) == 0:
        return None
    return tmp.iloc[0]


def pivot_metric(summary, metric, labels=EPS_ORDER):
    out = {}
    for framework in ["FL", "DFL"]:
        vals = []
        for lab in labels:
            row = get_row(summary, framework, lab)
            vals.append(np.nan if row is None else row.get(metric, np.nan))
        out[framework] = vals
    return out


def load_inputs():
    summary = pd.read_csv(SUMMARY_CSV)
    fl_rounds = pd.read_csv(FL_ROUNDS_CSV)
    dfl_rounds = pd.read_csv(DFL_ROUNDS_CSV)

    summary = prepare_summary(summary)

    rounds = pd.concat([fl_rounds, dfl_rounds], ignore_index=True)
    rounds["Setting Label"] = rounds.apply(label_from_row, axis=1)

    return summary, rounds


def title_then_legend(ax, title, ncol=2):
    ax.set_title(title, y=TITLE_Y)
    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, LEGEND_Y),
        ncol=ncol,
        frameon=True,
        borderaxespad=0.0
    )


def plot_line_pair(ax, x, y_fl, y_dfl, title, xlabel, ylabel,
                   random_line=None, xlabels=None, legend_ncol=2):
    ax.plot(x, y_fl, FL_STYLE, color=FL_COLOR, marker=FL_MARKER,
            linewidth=2, markersize=5, label="FL")
    ax.plot(x, y_dfl, DFL_STYLE, color=DFL_COLOR, marker=DFL_MARKER,
            linewidth=2, markersize=5, label="DFL")

    if random_line is not None:
        ax.axhline(random_line, color="black", linestyle=":", linewidth=1.2, label="Random guessing")
        legend_ncol = 3

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    if xlabels is not None:
        ax.set_xticks(x)
        ax.set_xticklabels(xlabels, rotation=35, ha="right")

    title_then_legend(ax, title, ncol=legend_ncol)


def plot_bar_pair(ax, labels, fl_vals, dfl_vals, title, ylabel, ymax=None):
    x = np.arange(len(labels))
    width = 0.38

    fl_plot = [0 if pd.isna(v) else v for v in fl_vals]
    dfl_plot = [0 if pd.isna(v) else v for v in dfl_vals]

    ax.bar(x - width / 2, fl_plot, width, color=FL_COLOR, label="FL")
    ax.bar(x + width / 2, dfl_plot, width, color=DFL_COLOR, label="DFL")

    if ymax is not None:
        ax.set_ylim(0, ymax)
        ax.axhline(ymax, color="black", linestyle=":", linewidth=1.0)

    for i, v in enumerate(fl_vals):
        if pd.isna(v):
            ax.text(i - width / 2, 1, "Never", rotation=90, ha="center", va="bottom", fontsize=8)
    for i, v in enumerate(dfl_vals):
        if pd.isna(v):
            ax.text(i + width / 2, 1, "Never", rotation=90, ha="center", va="bottom", fontsize=8)

    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=35, ha="right")
    title_then_legend(ax, title, ncol=2)


def adjust_single_panel(fig):
    fig.subplots_adjust(top=0.74, bottom=0.18, left=0.12, right=0.97)


def adjust_three_panel(fig):
    fig.subplots_adjust(top=0.70, bottom=0.22, wspace=0.35)


def adjust_six_panel(fig):
    fig.subplots_adjust(top=0.84, bottom=0.12, hspace=0.74, wspace=0.30)


# ============================================================
# DATA DISTRIBUTION
# ============================================================
def fig_data_distribution_only():
    if os.path.exists(DATA_DIST_IMAGE):
        dst_png = os.path.join(OUT_DIR, "Fig0_MNIST_data_distribution.png")
        shutil.copyfile(DATA_DIST_IMAGE, dst_png)
        print("Saved:", dst_png)
        return

    print("Data distribution image not found:", DATA_DIST_IMAGE)
    print("Fallback: generating distribution from CSV if available.")
    fig, ax = plt.subplots(figsize=(14, 4.5))
    draw_distribution_from_csv(ax)
    save_fig(fig, "Fig0_MNIST_data_distribution")


def draw_distribution_from_csv(ax):
    if os.path.exists(DATA_DIST_CSV):
        dist = pd.read_csv(DATA_DIST_CSV)
        cols_lower = {c.lower(): c for c in dist.columns}

        if "node" in cols_lower and all(f"C{i}" in dist.columns for i in range(10)):
            node_col = cols_lower["node"]
            bottom = np.zeros(len(dist))
            for i in range(10):
                y = dist[f"C{i}"].values
                ax.bar(dist[node_col], y, bottom=bottom, label=f"C{i}", width=0.85)
                bottom += y

        elif {"node", "class", "count"}.issubset(set(cols_lower)):
            node_col = cols_lower["node"]
            class_col = cols_lower["class"]
            count_col = cols_lower["count"]
            pivot = dist.pivot_table(index=node_col, columns=class_col, values=count_col,
                                     aggfunc="sum", fill_value=0)
            bottom = np.zeros(len(pivot))
            for c in sorted(pivot.columns):
                y = pivot[c].values
                ax.bar(pivot.index, y, bottom=bottom, label=f"C{c}", width=0.85)
                bottom += y
        else:
            ax.text(0.5, 0.5, "Data distribution CSV format not recognized", ha="center", va="center")
    else:
        ax.text(0.5, 0.5, "Data distribution image/CSV not found", ha="center", va="center")

    ax.set_title("Data Distribution - MNIST (Dirichlet α=0.5, Nodes=100)", y=TITLE_Y)
    ax.set_xlabel("Node")
    ax.set_ylabel("Number of samples")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, LEGEND_Y), ncol=10,
              frameon=True, borderaxespad=0.0)


# ============================================================
# MAIN FIGURES
# ============================================================
def fig_convergence(summary, rounds):
    fig, ax = plt.subplots(figsize=(8, 4.2))

    for fw, color, marker, style in [("FL", FL_COLOR, FL_MARKER, FL_STYLE), ("DFL", DFL_COLOR, DFL_MARKER, DFL_STYLE)]:
        sub = rounds[(rounds["Framework"] == fw) & (rounds["Setting Label"] == "NoDP")]
        ax.plot(sub["Round"], sub["Accuracy"], style, color=color, marker=marker,
                markevery=10, linewidth=2, markersize=4, label=fw)

    ax.set_xlabel("Communication Round")
    ax.set_ylabel("Test Accuracy")
    title_then_legend(ax, "(a) MNIST Learning Convergence Without DP", ncol=2)
    adjust_single_panel(fig)
    save_fig(fig, "Fig1_MNIST_convergence_NoDP")


def fig_convergence_and_distribution(summary, rounds):
    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.25], hspace=0.60)

    ax1 = fig.add_subplot(gs[0, 0])
    for fw, color, marker, style in [("FL", FL_COLOR, FL_MARKER, FL_STYLE), ("DFL", DFL_COLOR, DFL_MARKER, DFL_STYLE)]:
        sub = rounds[(rounds["Framework"] == fw) & (rounds["Setting Label"] == "NoDP")]
        ax1.plot(sub["Round"], sub["Accuracy"], style, color=color, marker=marker,
                 markevery=10, linewidth=2, markersize=4, label=fw)
    ax1.set_xlabel("Communication Round")
    ax1.set_ylabel("Test Accuracy")
    title_then_legend(ax1, "(a) MNIST Learning Convergence Without DP", ncol=2)

    ax2 = fig.add_subplot(gs[1, 0])
    if os.path.exists(DATA_DIST_IMAGE):
        img = mpimg.imread(DATA_DIST_IMAGE)
        ax2.imshow(img)
        ax2.axis("off")
        ax2.set_title("(b) MNIST Non-IID Data Distribution", y=1.05)
    else:
        draw_distribution_from_csv(ax2)
        ax2.set_title("(b) MNIST Non-IID Data Distribution", y=TITLE_Y)

    fig.subplots_adjust(top=0.88, bottom=0.06)
    save_fig(fig, "Fig1_MNIST_convergence_and_distribution")


def fig_rounds_to_threshold(summary):
    vals = pivot_metric(summary, "Rounds-to-Threshold", EPS_ORDER)
    fig, ax = plt.subplots(figsize=(8, 4.2))
    plot_bar_pair(ax, EPS_ORDER, vals["FL"], vals["DFL"], "(a) MNIST Rounds to Threshold", "Rounds to Threshold", ymax=100)
    adjust_single_panel(fig)
    save_fig(fig, "Fig2_MNIST_rounds_to_threshold")


def fig_final_accuracy(summary):
    labels = [f"ε={e:g}" for e in EPS_NUMERIC]
    vals = pivot_metric(summary, "Final Accuracy", labels)
    fig, ax = plt.subplots(figsize=(8, 4.2))
    plot_line_pair(ax, EPS_NUMERIC, vals["FL"], vals["DFL"], "(a) MNIST Accuracy Under DP Budgets", r"Privacy Budget $\epsilon$", "Final Accuracy")
    adjust_single_panel(fig)
    save_fig(fig, "Fig3_MNIST_accuracy_under_dp")


def fig_utility_retention(summary):
    labels = [f"ε={e:g}" for e in EPS_NUMERIC]
    values = {"FL": [], "DFL": []}

    for fw in ["FL", "DFL"]:
        base = get_row(summary, fw, "NoDP")
        base_acc = np.nan if base is None else base["Final Accuracy"]
        for lab in labels:
            row = get_row(summary, fw, lab)
            acc = np.nan if row is None else row["Final Accuracy"]
            values[fw].append(acc / base_acc if base_acc and not pd.isna(acc) else np.nan)

    fig, ax = plt.subplots(figsize=(8, 4.2))
    plot_line_pair(ax, EPS_NUMERIC, values["FL"], values["DFL"], "(a) MNIST Utility Retention", r"Privacy Budget $\epsilon$", "Utility Retention")
    ax.axhline(1.0, color="black", linestyle=":", linewidth=1.0)
    adjust_single_panel(fig)
    save_fig(fig, "Fig4_MNIST_utility_retention")


def fig_accuracy_degradation(summary):
    labels = [f"ε={e:g}" for e in EPS_NUMERIC]
    values = {"FL": [], "DFL": []}

    for fw in ["FL", "DFL"]:
        base = get_row(summary, fw, "NoDP")
        base_acc = np.nan if base is None else base["Final Accuracy"]
        for lab in labels:
            row = get_row(summary, fw, lab)
            acc = np.nan if row is None else row["Final Accuracy"]
            values[fw].append(base_acc - acc if not pd.isna(acc) and not pd.isna(base_acc) else np.nan)

    fig, ax = plt.subplots(figsize=(8, 4.2))
    plot_line_pair(ax, EPS_NUMERIC, values["FL"], values["DFL"], "(a) MNIST Accuracy Degradation", r"Privacy Budget $\epsilon$", "Accuracy Drop from No-DP")
    ax.axhline(0.0, color="black", linestyle=":", linewidth=1.0)
    adjust_single_panel(fig)
    save_fig(fig, "Fig5_MNIST_accuracy_degradation")


# ============================================================
# MIA FIGURES
# ============================================================
def fig_mia_auc_all(summary):
    labels = EPS_ORDER
    metrics = [("Confidence Attack AUC", "MIA AUC (Confidence)"), ("Loss Attack AUC", "MIA AUC (Loss)"), ("Entropy Attack AUC", "MIA AUC (Entropy)")]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    x = np.arange(len(labels))

    for idx, (ax, (metric, title)) in enumerate(zip(axes, metrics)):
        vals = pivot_metric(summary, metric, labels)
        ax.plot(x, vals["FL"], FL_STYLE, color=FL_COLOR, marker=FL_MARKER, linewidth=2, markersize=5, label="FL")
        ax.plot(x, vals["DFL"], DFL_STYLE, color=DFL_COLOR, marker=DFL_MARKER, linewidth=2, markersize=5, label="DFL")
        ax.axhline(0.5, color="black", linestyle=":", linewidth=1.1, label="Random guessing")
        ax.set_xlabel("Privacy Setting")
        ax.set_ylabel("Attack AUC")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35, ha="right")
        title_then_legend(ax, f"({chr(97 + idx)}) {title}", ncol=3)

        all_vals = np.array(vals["FL"] + vals["DFL"], dtype=float)
        all_vals = all_vals[~np.isnan(all_vals)]
        if len(all_vals) > 0:
            ymin = max(0.45, min(0.5, all_vals.min() - 0.02))
            ymax = min(0.60, max(0.5, all_vals.max() + 0.02))
            ax.set_ylim(ymin, ymax)

    fig.suptitle("Membership Inference Attack AUC Across Attack Variants", y=0.995, fontsize=13)
    adjust_three_panel(fig)
    save_fig(fig, "Fig6_MNIST_all_MIA_AUC")


def fig_mia_auc_strongest(summary):
    vals = pivot_metric(summary, "Strongest Membership Inference Attack (MIA) AUC", EPS_ORDER)
    x = np.arange(len(EPS_ORDER))
    fig, ax = plt.subplots(figsize=(8, 4.2))
    plot_line_pair(ax, x, vals["FL"], vals["DFL"], "Strongest Membership Inference Attack (MIA) AUC", "Privacy Setting", "Attack AUC", random_line=0.5, xlabels=EPS_ORDER, legend_ncol=3)
    adjust_single_panel(fig)
    save_fig(fig, "Fig6b_MNIST_strongest_MIA_AUC")


# ============================================================
# STABILITY
# ============================================================
def fig_stability(summary):
    vals = pivot_metric(summary, "Stability Round", EPS_ORDER)
    fig, ax = plt.subplots(figsize=(8, 4.2))
    plot_bar_pair(ax, EPS_ORDER, vals["FL"], vals["DFL"], "(a) MNIST Stability Analysis", "Stability Round", ymax=100)
    adjust_single_panel(fig)
    save_fig(fig, "Fig7_MNIST_stability_round")


# ============================================================
# GRADIENT ATTACK FIGURES
# ============================================================
def fig_gradient_invgrad_metrics(summary):
    labels = EPS_ORDER
    metrics = [
        ("invGrad MSE", "MSE", "Lower values indicate stronger reconstruction"),
        ("invGrad PSNR", "PSNR", "Higher values indicate stronger reconstruction"),
        ("invGrad SSIM", "SSIM", "Higher values indicate stronger structural similarity"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
    x = np.arange(len(labels))

    for idx, (ax, (metric, ylabel, footer_text)) in enumerate(zip(axes, metrics)):
        vals = pivot_metric(summary, metric, labels)

        ax.plot(
            x, vals["FL"],
            FL_STYLE,
            color=FL_COLOR,
            marker=FL_MARKER,
            linewidth=2,
            markersize=5,
            label="FL"
        )

        ax.plot(
            x, vals["DFL"],
            DFL_STYLE,
            color=DFL_COLOR,
            marker=DFL_MARKER,
            linewidth=2,
            markersize=5,
            label="DFL"
        )

        ax.set_ylabel(ylabel)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35, ha="right")

        title_then_legend(
            ax,
            f"({chr(97 + idx)}) {metric}",
            ncol=2
        )

        ax.text(
            0.5,
            -0.24,
            footer_text,
            transform=ax.transAxes,
            ha="center",
            va="top",
            fontsize=9
        )

    fig.suptitle(
        "Gradient Inversion Attack Reconstruction Quality",
        y=0.995,
        fontsize=13
    )

    fig.subplots_adjust(
        top=0.66,
        bottom=0.28,
        wspace=0.35
    )

    save_fig(fig, "Fig8_MNIST_invGrad_reconstruction_metrics")



def fig_gradient_all_attacks_mse_ssim(summary):
    labels = EPS_ORDER
    attacks = ["DLG", "iDLG", "invGrad"]
    fig, axes = plt.subplots(2, 3, figsize=(15, 7.4), sharex=True)
    x = np.arange(len(labels))

    for j, attack in enumerate(attacks):
        for i, metric_name in enumerate(["MSE", "SSIM"]):
            metric = f"{attack} {metric_name}"
            ax = axes[i, j]
            vals = pivot_metric(summary, metric, labels)
            ax.plot(x, vals["FL"], FL_STYLE, color=FL_COLOR, marker=FL_MARKER, linewidth=2, markersize=5, label="FL")
            ax.plot(x, vals["DFL"], DFL_STYLE, color=DFL_COLOR, marker=DFL_MARKER, linewidth=2, markersize=5, label="DFL")
            ax.set_ylabel(metric_name)
            ax.set_xticks(x)
            ax.set_xticklabels(labels, rotation=35, ha="right")
            title_then_legend(ax, f"({chr(97 + i * 3 + j)}) {attack} {metric_name}", ncol=2)

    fig.suptitle("Gradient Inversion Attack Metrics Across Attack Types", y=0.995, fontsize=13)
    adjust_six_panel(fig)
    save_fig(fig, "Fig9_MNIST_gradient_attacks_MSE_SSIM")


# ============================================================
# RUN
# ============================================================
def make_all_figures():
    summary, rounds = load_inputs()

    clean_path = os.path.join(OUT_DIR, "summary_with_derived_metrics.csv")
    summary.to_csv(clean_path, index=False)
    print("Saved:", clean_path)

    fig_data_distribution_only()
    fig_convergence(summary, rounds)
    fig_convergence_and_distribution(summary, rounds)
    fig_rounds_to_threshold(summary)
    fig_final_accuracy(summary)
    fig_utility_retention(summary)
    fig_accuracy_degradation(summary)
    fig_mia_auc_all(summary)
    fig_mia_auc_strongest(summary)
    fig_stability(summary)
    fig_gradient_invgrad_metrics(summary)
    fig_gradient_all_attacks_mse_ssim(summary)

    print("\nAll figures saved in:", OUT_DIR)


if __name__ == "__main__":
    make_all_figures()


Saved: C:\Users\Alsha\Desktop\Last EXP 15-6-2026\MNIST_Final_Figures_Journal_Style\summary_with_derived_metrics.csv
Saved: C:\Users\Alsha\Desktop\Last EXP 15-6-2026\MNIST_Final_Figures_Journal_Style\Fig0_MNIST_data_distribution.png
Saved: C:\Users\Alsha\Desktop\Last EXP 15-6-2026\MNIST_Final_Figures_Journal_Style\Fig1_MNIST_convergence_NoDP.png
Saved: C:\Users\Alsha\Desktop\Last EXP 15-6-2026\MNIST_Final_Figures_Journal_Style\Fig1_MNIST_convergence_NoDP.pdf
Saved: C:\Users\Alsha\Desktop\Last EXP 15-6-2026\MNIST_Final_Figures_Journal_Style\Fig1_MNIST_convergence_and_distribution.png
Saved: C:\Users\Alsha\Desktop\Last EXP 15-6-2026\MNIST_Final_Figures_Journal_Style\Fig1_MNIST_convergence_and_distribution.pdf
Saved: C:\Users\Alsha\Desktop\Last EXP 15-6-2026\MNIST_Final_Figures_Journal_Style\Fig2_MNIST_rounds_to_threshold.png
Saved: C:\Users\Alsha\Desktop\Last EXP 15-6-2026\MNIST_Final_Figures_Journal_Style\Fig2_MNIST_rounds_to_threshold.pdf
Saved: C:\Users\Alsha\Desktop\Last EXP 15-6-2026